# INITIAL CONFIGURATION

In [2]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

# %load_ext autoreload
# %autoreload 2
%matplotlib inline

/home/turbotowerlnx/Documents/Master/SMA/SMA-MultiAgent-Based-Crowd-Simulation/notebooks
/home/turbotowerlnx/Documents/Master/SMA/SMA-MultiAgent-Based-Crowd-Simulation/app


# ANALYSIS

In [30]:
from maikol_utils.file_utils import list_dir_files

from src.config import Configuration

CONFIG = Configuration()

experiments, _ = list_dir_files(CONFIG.LOGS_DIR)

experiments = [exp for exp in experiments if ".csv" in exp]
experiments

['../logs/experiments_results_evaluate_different_agent_densities.csv']

In [31]:
df = pd.read_csv(experiments[0])


df.groupby(["RunId", "iteration", "initial_agents"])["Step"].max()
# df.head(200)

RunId  iteration  initial_agents
0      0          10                 71
1      0          25                 75
2      0          50                 74
3      0          75                 85
4      0          100               119
                                   ... 
7995   999        75                 92
7996   999        100                79
7997   999        150                84
7998   999        200               114
7999   999        300               424
Name: Step, Length: 8000, dtype: int64

In [35]:

# Get max steps per run
max_steps_per_run = df.groupby(["RunId", "initial_agents"])["Step"].mean().reset_index()
max_steps_per_run.columns = ["RunId", "initial_agents", "max_steps"]
max_steps_per_run.head(10)


,RunId,initial_agents,max_steps
0,0,10,35.5
1,1,25,37.5
2,2,50,37.0
3,3,75,42.5
4,4,100,59.5
5,5,150,41.5
6,6,200,121.5
7,7,300,219.5
8,8,10,30.5
9,9,25,28.5


In [33]:

# Compute statistics per experiment type (initial_agents)
stats_per_experiment = max_steps_per_run.groupby("initial_agents")["max_steps"].agg([
    "count",      # number of runs
    "mean",       # mean max steps
    "median",     # median max steps
    "std",        # standard deviation
    "min",        # minimum max steps
    "max",        # maximum max steps
    ("q25", lambda x: x.quantile(0.25)),  # 25th percentile
    ("q75", lambda x: x.quantile(0.75)),  # 75th percentile
]).round(2)

print("Statistics of Max Steps per Experiment Type:")
print(stats_per_experiment)


Statistics of Max Steps per Experiment Type:
                count    mean  median    std  min  max    q25     q75
initial_agents                                                       
10               1000   60.18    60.0  10.89   31   95   53.0   67.00
25               1000   67.59    67.0   9.79   42  116   61.0   74.00
50               1000   73.53    73.0   8.35   53  116   68.0   78.25
75               1000   77.06    76.0   8.99   56  184   71.0   82.00
100              1000   81.78    80.0  11.15   61  180   75.0   86.00
150              1000  106.22    93.0  36.20   71  437   85.0  112.00
200              1000  165.21   151.0  57.42   82  421  124.0  192.00
300              1000  407.50   404.0  77.42  203  671  350.0  459.00


In [37]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots: box plot and line plot with error bars
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribution of Max Steps by Initial Agents", 
                    "Mean Max Steps by Initial Agents (with Std Dev)"),
    specs=[[{}, {}]]
)

# Box plot
for agent_count in sorted(max_steps_per_run["initial_agents"].unique()):
    data = max_steps_per_run[max_steps_per_run["initial_agents"] == agent_count]["max_steps"]
    fig.add_trace(
        go.Box(y=data, name=f"{agent_count}", showlegend=False),
        row=1, col=1
    )

# Line plot with error bars
means = stats_per_experiment["mean"]
stds = stats_per_experiment["std"]
fig.add_trace(
    go.Scatter(
        x=means.index,
        y=means.values,
        error_y=dict(type="data", array=stds.values, visible=True),
        mode="lines+markers",
        name="Mean ± Std",
        marker=dict(size=8),
        line=dict(width=2)
    ),
    row=1, col=2
)

# Update layout
fig.update_xaxes(title_text="Initial Agents", row=1, col=1)
fig.update_yaxes(title_text="Max Steps", row=1, col=1)
fig.update_xaxes(title_text="Initial Agents", row=1, col=2)
fig.update_yaxes(title_text="Max Steps", row=1, col=2)

fig.update_layout(height=500, width=1200, showlegend=True)
fig.show()

# ...

In [ ]:
from src.config import Configuration

DEFAULT_CONFIG = Configuration(
    n_experiments=1000,
    initial_agents=75,
    width=30,
    height=30,
    agent_types_ratios = {
        "polite": 0.7,
        "aggressive": 0.2,
        "slow": 0.1
    },
    track_last_steps=4,
    path_finding_algorithm="A*",
    differentiate_exits=True,
    respawn_agents=False,
    scenario_type="MALL",
    n_exits=4,
)

### Batch Run

### Alternative: Multiprocessing-safe batch run
If you need multiple processes, disable autoreload first and restart the kernel:

In [ ]:
EXPERIMENTS = [
    {
        "title": "...",
        "batch_params": {
            "width": DEFAULT_CONFIG.width,
            "height": DEFAULT_CONFIG.height,
            "initial_agents": [10, 25, 50, 75, 100, 150, 200, 300],
            "track_last_steps": DEFAULT_CONFIG.track_last_steps,
            "path_finding_algorithm": DEFAULT_CONFIG.path_finding_algorithm,
            "differentiate_exits": DEFAULT_CONFIG.differentiate_exits,
            "respawn_agents": DEFAULT_CONFIG.respawn_agents,
            "polite_ratio": 1.0,
            "aggressive_ratio": 0.0,
            "slow_ratio": 0.0,
            "scenario_type": DEFAULT_CONFIG.scenario_type,
            "n_exits": DEFAULT_CONFIG.n_exits,
        }
    }
]

In [4]:
import mesa 
import sys
import numpy as np
import pandas as pd
from maikol_utils.print_utils import print_separator, print_color

from src.models import CrowdModelWrapper

for EXP in EXPERIMENTS:
    print_separator(f"Running configuration {EXP['title']}", sep_type="START")
    rng = np.random.default_rng(DEFAULT_CONFIG.seed)
    seed_values = rng.integers(0, sys.maxsize, size=(DEFAULT_CONFIG.n_experiments,))

    results = mesa.batch_run(
        model_cls=CrowdModelWrapper,
        parameters=EXP['batch_params'],
        rng=seed_values.tolist(),
        max_steps=EXP.get("max_steps", 2000),
        number_processes=DEFAULT_CONFIG.n_processes,
        data_collection_period=1,
        display_progress=True,
    )

    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(DEFAULT_CONFIG.LOGS_DIR, f"experiments_results_{EXP['title'].replace(' ', '_').lower()}.csv"), index=False)



                                                   Running configuration ...                                                    




  0%|          | 0/8000 [00:00<?, ?it/s]

Process SpawnPoolWorker-20:
Process SpawnPoolWorker-18:
Process SpawnPoolWorker-16:
Process SpawnPoolWorker-12:
Process SpawnPoolWorker-10:
Process SpawnPoolWorker-13:
Process SpawnPoolWorker-19:
Process SpawnPoolWorker-15:
Process SpawnPoolWorker-14:
Process SpawnPoolWorker-17:
Process SpawnPoolWorker-11:
Process SpawnPoolWorker-4:
Process SpawnPoolWorker-6:
Process SpawnPoolWorker-1:
Process SpawnPoolWorker-7:
Process SpawnPoolWorker-3:
Process SpawnPoolWorker-9:
Process SpawnPoolWorker-8:
Process SpawnPoolWorker-2:
Process SpawnPoolWorker-5:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call l

KeyboardInterrupt: 